Download the ClonedPerson Dataset images from the Google drive link in the https://github.com/Yanan-Wang-cs/ClonedPerson repository.

Extract the dataset into the directory structure:
```
.
├── train
│   │   ├── 700000_s02_c00_f023389.jpg
│   │   ├── 700000_s06_c00_f023413.jpg
│   │   ├── ...
├── test
│   ├── gallery
│   │   ├── 704830_s02_c00_f011773.jpg
│   │   ├── 704830_s02_c00_f011797.jpg
│   │   ├── 704830_s02_c00_f011845.jpg
│   │   ├── ...
│   ├── query
│   │   │   ├── 705227_s02_c02_f001117.jpg
│   │   │   ├── 705624_s12_c00_f003229.jpg
│   │   │   ├── ...
```
Run the following cells to perform the finetuning.
Evaluation was skipped due to taking lots of time and memory.

In [ ]:
import os
import re
import glob
import os.path as osp
from torchreid.reid.data import ImageDataset

class ClonedPerson(ImageDataset):
    dataset_dir = 'ClonedPerson'

    def __init__(self, root='', **kwargs):
        self.root = osp.abspath(osp.expanduser(root))
        self.dataset_dir = osp.join(self.root, self.dataset_dir)
        
        # Define paths for your subsets
        self.train_dir = osp.join(self.dataset_dir, 'train')
        self.query_dir = osp.join(self.dataset_dir, 'test', 'query')
        self.gallery_dir = osp.join(self.dataset_dir, 'test', 'gallery')

        # Check if the directories exist
        self.check_before_run()

        # Process directories and extract (img_path, pid, camid)
        train = self.process_dir(self.train_dir, relabel=True)
        query = self.process_dir(self.query_dir, relabel=False)
        gallery = self.process_dir(self.gallery_dir, relabel=False)

        super(ClonedPerson, self).__init__(train, query, gallery, **kwargs)

    def check_before_run(self):
        """Check if directories exist."""
        if not osp.exists(self.dataset_dir):
            raise RuntimeError(f"'{self.dataset_dir}' is not found")
        if not osp.exists(self.train_dir):
            raise RuntimeError(f"'{self.train_dir}' is not found")
        if not osp.exists(self.query_dir):
            raise RuntimeError(f"'{self.query_dir}' is not found")
        if not osp.exists(self.gallery_dir):
            raise RuntimeError(f"'{self.gallery_dir}' is not found")

    def process_dir(self, dir_path, relabel=False):
        img_paths = glob.glob(osp.join(dir_path, '*.jpg'))
        pattern = re.compile(r'([\d]+)_s([\d]+)_c([\d]+)_f([\d]+)')

        # Initialize dictionary to track uniquely mapping global IDs
        pid_container = set()
        for img_path in img_paths:
            filename = osp.basename(img_path)
            pid, _, _, _ = map(int, pattern.search(filename).groups())
            pid_container.add(pid)
        
        pid2label = {pid: label for label, pid in enumerate(sorted(pid_container))}

        data = []
        for img_path in img_paths:
            filename = osp.basename(img_path)
            pid, _, camid, _ = map(int, pattern.search(filename).groups())
            
            if relabel:
                pid = pid2label[pid]
            
            data.append((img_path, pid, camid))

        return data

/home/user/miniconda3/envs/reid_wsl/lib/python3.12/site-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [ ]:
import torchreid

# 1. Register the custom dataset
torchreid.data.register_image_dataset('clonedperson', ClonedPerson)

In [ ]:
DATASET_ROOT = './'

# 2. Setup the DataManager
datamanager = torchreid.data.ImageDataManager(
    root=DATASET_ROOT,
    sources='clonedperson',
    targets='clonedperson',
    height=256,
    width=128,
    batch_size_train=128,
    batch_size_test=1,
    workers=8,
)

Building train transforms ...
+ resize to 256x128
+ random flip
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
Building test transforms ...
+ resize to 256x128
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
=> Loading train (source) dataset
=> Loaded ClonedPerson
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |  4826 |   763953 |         4
  query    |   795 |    12394 |         4
  gallery  |   795 |   111419 |         4
  ----------------------------------------
=> Loading test (target) dataset
=> Loaded ClonedPerson
  ----------------------------------------
  subset   | # ids | # images | # cameras
  ----------------------------------------
  train    |  4826 |   763953 |         4
  query    |   795 |    12394 |         4
  gallery  |   795 |   111419 |         4
  ---------

In [ ]:
# Fix for PyTorch 2.6 or above

import pickle
from functools import partial
from collections import OrderedDict

def load_checkpoint(fpath):
    r"""Loads checkpoint.

    ``UnicodeDecodeError`` can be well handled, which means
    python2-saved files can be read from python3.

    Args:
        fpath (str): path to checkpoint.

    Returns:
        dict

    Examples::
        >>> from torchreid.utils import load_checkpoint
        >>> fpath = 'log/my_model/model.pth.tar-10'
        >>> checkpoint = load_checkpoint(fpath)
    """
    if fpath is None:
        raise ValueError('File path is None')
    fpath = osp.abspath(osp.expanduser(fpath))
    if not osp.exists(fpath):
        raise FileNotFoundError('File is not found at "{}"'.format(fpath))
    map_location = None if torch.cuda.is_available() else 'cpu'
    try:
        checkpoint = torch.load(fpath, map_location=map_location, weights_only=False)
    except UnicodeDecodeError:
        pickle.load = partial(pickle.load, encoding="latin1")
        pickle.Unpickler = partial(pickle.Unpickler, encoding="latin1")
        checkpoint = torch.load(
            fpath, pickle_module=pickle, map_location=map_location,weights_only=False
        )
    except Exception:
        print('Unable to load checkpoint from "{}"'.format(fpath))
        raise
    return checkpoint

def load_pretrained_weights(model, weight_path):
    r"""Loads pretrianed weights to model.

    Features::
        - Incompatible layers (unmatched in name or size) will be ignored.
        - Can automatically deal with keys containing "module.".

    Args:
        model (nn.Module): network model.
        weight_path (str): path to pretrained weights.

    Examples::
        >>> from torchreid.utils import load_pretrained_weights
        >>> weight_path = 'log/my_model/model-best.pth.tar'
        >>> load_pretrained_weights(model, weight_path)
    """
    checkpoint = load_checkpoint(weight_path)
    if 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint

    model_dict = model.state_dict()
    new_state_dict = OrderedDict()
    matched_layers, discarded_layers = [], []

    for k, v in state_dict.items():
        if k.startswith('module.'):
            k = k[7:] # discard module.

        if k in model_dict and model_dict[k].size() == v.size():
            new_state_dict[k] = v
            matched_layers.append(k)
        else:
            discarded_layers.append(k)

    model_dict.update(new_state_dict)
    model.load_state_dict(model_dict)

    if len(matched_layers) == 0:
        print.warn(
            'The pretrained weights "{}" cannot be loaded, '
            'please check the key names manually '
            '(** ignored and continue **)'.format(weight_path)
        )
    else:
        print(
            'Successfully loaded pretrained weights from "{}"'.
            format(weight_path)
        )
        if len(discarded_layers) > 0:
            print(
                '** The following layers are discarded '
                'due to unmatched keys or layer size: {}'.
                format(discarded_layers)
            )

In [ ]:
import torchreid
import torch
# ==========================================
# 3. BUILD MODEL & CONFIGURE TRAINING
# ==========================================
# Build the OSNet architecture (dynamically matching your subset's unique person count)
model = torchreid.models.build_model(
    name='osnet_x1_0',
    num_classes=datamanager.num_train_pids,
    loss='softmax',
    pretrained=False  # Warm start using ImageNet weights
)
load_pretrained_weights(model, './weights/osnet_ms_d_c.pth.tar')

# load_pretrained_weights(model, 'osnet_x1_0_msmt17_combineall_256x128_amsgrad_ep150_stp60_lr0.0015_b64_fb10_softmax_labelsmooth_flip_jitter.pth')

if torch.cuda.is_available():
    model = model.cuda()

# Set up optimized Adam hyperparameters standard for OSNet training
optimizer = torchreid.optim.build_optimizer(
    model,
    optim='adam',
    lr=0.0005
)

scheduler = torchreid.optim.build_lr_scheduler(
    optimizer,
    lr_scheduler='single_step',
    stepsize=20,
    gamma=0.1
)

# Initialize the training loop wrapper
engine = torchreid.engine.ImageSoftmaxEngine(
    datamanager,
    model,
    optimizer=optimizer,
    scheduler=scheduler,
    label_smooth=True
)


Successfully loaded pretrained weights from "./weights/osnet_ms_d_c.pth.tar"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']


In [ ]:
# ==========================================
# 4. EXECUTE TRAINING
# ==========================================
engine.run(
    max_epoch=25,
    save_dir='log/osnet_clonedperson',
    start_eval=31,
    test_only=False,  # Tells engine to run training epochs, not just test cycles
    ranks=[1,5]
)

=> Start training
epoch: [1/25][10/5968]	time 0.299 (0.526)	data 0.001 (0.089)	eta 21:47:36	loss 8.6606 (8.5433)	acc 0.0000 (0.0000)	lr 0.000600
epoch: [1/25][20/5968]	time 0.316 (0.422)	data 0.001 (0.045)	eta 17:30:26	loss 8.8792 (8.6555)	acc 0.0000 (0.0781)	lr 0.000600
epoch: [1/25][30/5968]	time 0.340 (0.388)	data 0.001 (0.030)	eta 16:05:01	loss 8.9989 (8.7310)	acc 0.0000 (0.1562)	lr 0.000600
epoch: [1/25][40/5968]	time 0.310 (0.371)	data 0.001 (0.023)	eta 15:21:18	loss 8.8337 (8.7656)	acc 0.0000 (0.1367)	lr 0.000600
epoch: [1/25][50/5968]	time 0.302 (0.358)	data 0.001 (0.018)	eta 14:49:51	loss 8.8147 (8.7953)	acc 0.0000 (0.1719)	lr 0.000600
epoch: [1/25][60/5968]	time 0.307 (0.322)	data 0.001 (0.015)	eta 13:20:39	loss 8.7987 (8.8051)	acc 0.0000 (0.2214)	lr 0.000600
epoch: [1/25][70/5968]	time 0.338 (0.323)	data 0.001 (0.013)	eta 13:23:06	loss 8.8595 (8.8066)	acc 0.0000 (0.2232)	lr 0.000600
epoch: [1/25][80/5968]	time 0.319 (0.322)	data 0.001 (0.012)	eta 13:20:53	loss 8.6983 (8.7949

In [ ]:
torch.save(model.state_dict(), 'log/osnet_clonedperson.pth')